In [11]:
! pip install joblib pandas scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import joblib
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesRegressor

print("Libraries imported successfully!")

Libraries imported successfully!


In [13]:
models_path = Path("../models")

models_path.mkdir(
    parents=True,
    exist_ok=True
)

print("Models folder ready:")
print(models_path.resolve())

Models folder ready:
C:\Users\rajum\OneDrive\Desktop\Smart_Electricity_Consumption\models


In [14]:
dataset_path = Path("../dataset/electricity_consumption_cleaned.csv")

df = pd.read_csv(dataset_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print()
print("Columns:")
print(df.columns.tolist())

Dataset loaded successfully!
Dataset shape: (19735, 29)

Columns:
['date', 'Appliances', 'lights', 'T1', 'RH_1', 'T2', 'RH_2', 'T3', 'RH_3', 'T4', 'RH_4', 'T5', 'RH_5', 'T6', 'RH_6', 'T7', 'RH_7', 'T8', 'RH_8', 'T9', 'RH_9', 'T_out', 'Press_mm_hg', 'RH_out', 'Windspeed', 'Visibility', 'Tdewpoint', 'rv1', 'rv2']


In [15]:
# Target
y = df["Appliances"]

# Features
X = df.drop("Appliances", axis=1)

# Keep numeric features only
X = X.select_dtypes(include=["number"])

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print()
print("Number of features:", X.shape[1])

print()
print("Feature names:")
for i, feature in enumerate(X.columns, start=1):
    print(f"{i}. {feature}")

Features shape: (19735, 27)
Target shape: (19735,)

Number of features: 27

Feature names:
1. lights
2. T1
3. RH_1
4. T2
5. RH_2
6. T3
7. RH_3
8. T4
9. RH_4
10. T5
11. RH_5
12. T6
13. RH_6
14. T7
15. RH_7
16. T8
17. RH_8
18. T9
19. RH_9
20. T_out
21. Press_mm_hg
22. RH_out
23. Windspeed
24. Visibility
25. Tdewpoint
26. rv1
27. rv2


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (15788, 27)
Testing data: (3947, 27)


In [17]:
final_model = ExtraTreesRegressor(
    n_estimators=50,
    max_depth=15,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=1
)

print("Training model...")

final_model.fit(
    X_train,
    y_train
)

print("Model trained successfully!")
print("Model type:", type(final_model).__name__)
print("Number of features:", final_model.n_features_in_)

Training model...
Model trained successfully!
Model type: ExtraTreesRegressor
Number of features: 27


In [18]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

predictions = final_model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = mean_squared_error(y_test, predictions) ** 0.5
r2 = r2_score(y_test, predictions)

print("MODEL PERFORMANCE")
print("-------------------------")
print("MAE :", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("R2  :", round(r2, 4))

MODEL PERFORMANCE
-------------------------
MAE : 33.3687
RMSE: 68.9735
R2  : 0.5246


In [19]:
model_file = models_path / "electricity_consumption_model.pkl"

joblib.dump(
    final_model,
    model_file,
    compress=3
)

model_size_mb = model_file.stat().st_size / (1024 * 1024)

print("Model saved successfully!")
print()
print("File:", model_file.resolve())
print("Model size:", round(model_size_mb, 2), "MB")

Model saved successfully!

File: C:\Users\rajum\OneDrive\Desktop\Smart_Electricity_Consumption\models\electricity_consumption_model.pkl
Model size: 4.06 MB


In [20]:
feature_file = models_path / "feature_columns.pkl"

joblib.dump(
    X_train.columns.tolist(),
    feature_file,
    compress=3
)

print("Feature columns saved successfully!")
print("File:", feature_file.resolve())

Feature columns saved successfully!
File: C:\Users\rajum\OneDrive\Desktop\Smart_Electricity_Consumption\models\feature_columns.pkl


In [21]:
print("FILES INSIDE MODELS FOLDER")
print("==========================")

for file in models_path.iterdir():
    if file.is_file():
        size_mb = file.stat().st_size / (1024 * 1024)
        print(f"{file.name} -> {size_mb:.2f} MB")

FILES INSIDE MODELS FOLDER
electricity_consumption_model.pkl -> 4.06 MB
feature_columns.pkl -> 0.00 MB


In [22]:
loaded_model = joblib.load(model_file)

loaded_features = joblib.load(feature_file)

print("Model loaded successfully!")
print("Model type:", type(loaded_model).__name__)
print("Number of features:", loaded_model.n_features_in_)

print()
print("Saved features:", len(loaded_features))

print()
print("All features:")
for i, feature in enumerate(loaded_features, start=1):
    print(f"{i}. {feature}")

Model loaded successfully!
Model type: ExtraTreesRegressor
Number of features: 27

Saved features: 27

All features:
1. lights
2. T1
3. RH_1
4. T2
5. RH_2
6. T3
7. RH_3
8. T4
9. RH_4
10. T5
11. RH_5
12. T6
13. RH_6
14. T7
15. RH_7
16. T8
17. RH_8
18. T9
19. RH_9
20. T_out
21. Press_mm_hg
22. RH_out
23. Windspeed
24. Visibility
25. Tdewpoint
26. rv1
27. rv2


In [23]:
sample = X_test.iloc[[0]]

result = loaded_model.predict(sample)[0]

print("Sample prediction successful!")
print("Predicted electricity consumption:",
      round(float(result), 2),
      "Wh")

Sample prediction successful!
Predicted electricity consumption: 52.17 Wh
